# Exploring ASTER Surface Kinetic Temperature (ASTER 08)

**Authors**: Nathan Roberts and Erik Bolch  
Last Updated: 2026-09-15

> &#128216; **Learning Objectives**
>
> 1. Search for [ASTER 08](https://doi.org/10.5067/ASTER/AST_08.004) granules using `earthaccess`
> 2. Open and Reproject ASTER 08 Data
> 3. Create and Visualize a Time Series

### Summary

This tutorial provides a practical guide for programmatically accessing [Advanced Spaceborne Thermal Emission and Reflection Radiometer (ASTER)](https://terra.nasa.gov/about/terra-instruments/aster) [Surface Kinetic Temperature (ASTER 08)](https://doi.org/10.5067/ASTER/AST_08.004) data using the `earthaccess` Python library and to create a large time series of data. You will learn how to retrieve ASTER observations, reproject them into a consistent north‑up orientation, and prepare them for time‑series analysis. The ASTER Reflectance and Radiance Level-2 (AST_07, AST_07XT, AST_09, AST_09XT, AST_09T, AST_05) products have a rotated geographic transformation. ASTER L1A Raw data are used as inputs to create the higher level products. Natively, the data is rotated at differing angles due to the orbital orientation of the Terra Satellite. The rotation value of each ASTER acquisition is noted within the Unified Metadata Model Granule (UMM-G) metadata `maporientationangle` field. In order to construct a time series stack of data, common tools require the data to be transformed to a rectilinear or North-Up transformation.

To efficiently reproject a large number of ASTER acquisitions, we will use `Dask` to take advantage of parallel computing. Once reprojected, the data can be combined into a multi‑temporal “stack” that is loaded into `rioxarray` for further geospatial processing. This includes clipping the dataset to the area surrounding the Kīlauea Volcano Crater in Hawaii (USA) and creating visualizations using the `holoviews` library.
By the end of the tutorial, you’ll have a reproducible workflow for building a thermal time series using ASTER 08 data.

### Background

The **Advanced Spaceborne Thermal Emission and Reflection Radiometer** (ASTER), launched on December 18, 1999 aboard NASA’s [Terra](https://terra.nasa.gov/) satellite, has collected more than three million unique images of Earth’s surface. ASTER acquires data at 15 to 90-meter resolution across 14 spectral bands spanning the visible to thermal infrared regions. One of its key products, Surface Kinetic Temperature (ASTER 08), is now archived as Cloud‑Optimized GeoTIFFs (COGs) within the Earthdata cloud environment. These cloud‑native formats, along with accompanying UMM‑C and UMM‑G metadata records, make the ASTER archive far more accessible to modern geospatial workflows.
With these updates, users can seamlessly search, authenticate, and retrieve ASTER 08 data using common Python tools—including NASA’s [`earthaccess`](https://www.earthdata.nasa.gov/data/tools/earthaccess) library. ASTER’s combination of high resolution thermal infrared imaging, broad spectral coverage, and a continuous multi‑decadal record makes it one of the most valuable satellite instruments for detecting, characterizing, and monitoring thermal activity.

### Prerequisites

- **NASA Earthdata Login** -  [Register Here](https://urs.earthdata.nasa.gov/users/new)
- **Compatible Python Environment** - Please refer to the [setup instructions](https://github.com/nasa/LPDAAC-Data-Resources/blob/main/setup/setup_instructions_python.md) in the [LPDAAC-Data-Resources repository](https://github.com/nasa/LPDAAC-Data-Resources)

### Data Used  

+ **ASTER 08 Surface Kinetic Temperature**([AST_08](https://doi.org/10.5067/ASTER/AST_08.004))
 
    - Variables (layers):  
        - [Surface kinetic temperature composite](https://cmr.earthdata.nasa.gov/search/concepts/V3489818670-LPCLOUD.html)<br>
 
 >**Important:** [ASTER Thermal Infrared (TIR) bands are no longer provided with data acquired after January 16, 2026](https://www.earthdata.nasa.gov/data/alerts-outages/terra-asters-tir-instrument-permanently-turned-off-vnir-data-collection-resumes)

 ### Tutorial Outline  
 
 1. [**Getting Started**](#getstarted)<br>
     1.1 Import Packages<br>
     1.2 Setup Current Working Directory<br>
     1.3 Earthdata Login Authentication<br>
 2. [**Searching ASTER 08 Version 4 Data Granules using `earthaccess`**](#find)<br>
     2.1 Define Our Query Parameters<br>
     2.2 Search for Data Granules for AST_08 Data Collection<br>
 3. [**Processing ASTER 08 Data**](#process)<br>
     3.1 Setup Helper Functions<br>
     3.2 Call the Helper Functions and Process Data<br>
 4. [**Post Processing ASTER 08 Data**](#post)<br>
     4.1 Filter out arrays that are None values<br>
     4.2 Grid Data Arrays to Common Grid and Create `xarray` Dataset<br>
     4.3 Visualize Subset of Results<br>
     4.4 Quality Filtering<br>
 5. [**Visualizations of ASTER 08 Data**](#visualizations)<br>
     5.1 Create Line Plot<br>
     5.2 Visualize Entire Time Series<br>

## 1. Getting Started<a id="getstarted"></a>

### 1.1 Import Packages 
Import the required packages.

In [ ]:
import rasterio as rio
from osgeo import gdal
import rioxarray as rxr
import os
import numpy as np
import earthaccess
import geopandas as gpd
import dask
from datetime import datetime
import pandas as pd
from rasterio.transform import from_origin
from rasterio.warp import Resampling
from dask.distributed import Client, LocalCluster
from collections import defaultdict
import xarray as xr
import matplotlib.pyplot as plt
from rioxarray.exceptions import NoDataInBounds
import hvplot.xarray
import geoviews as gv
import holoviews as hv
from holoviews import opts
from shapely.geometry import Point
import hvplot.pandas

hv.extension('bokeh')

### 1.2 Setup Current Working Directory
We will use a directory called `data` as the default working directory for the tutorial. Any required ancillary files will be read from there also any outputs downloaded will be written in this folder.

In [ ]:
os.chdir('../../data')

### 1.3 Earthdata Login Authentication

We will use the [earthaccess](https://github.com/nsidc/earthaccess#readme) package for authentication. `earthaccess` can either create a new local .netrc file to store credentials or validate that one exists already in your user profile. If you do not have a .netrc file, you will be prompted for your credentials and one will be created. 

In [ ]:
earthaccess.login(persist=True)

## 2. Searching ASTER 08 Version 4 Data Granules using `earthaccess` <a id="find"></a>
To find ASTER 08 Surface Kinetic Temperature data, we will use the `earthaccess` Python library to search [NASA's Common Metadata Repository (CMR)](https://www.earthdata.nasa.gov/about/esdis/eosdis/cmr). We will use a GeoJSON file containing our region of interest (ROI) to search for data granules that intersect the ROI. To do this, we will simplify this region to a bounding box and extract the bounding coordinates from the GeoPandas object after opening the file. We will also specify a date range in our search criteria.

### 2.1 Define Our Query Parameters
Here, we will read our GeoJSON file using GeoPandas. We will use the `total_bounds` property to get the bounding box of our ROI, and add that to a Python tuple, which is the `earthaccess.search_data` function's expected data type for the bounding_box parameter.

In [ ]:
aoi = gpd.read_file('kilauea.geojson')
bbox = tuple(list(aoi.total_bounds))

We can also search for data within a specific time period. In this example, we search from January 1, 2022, to December 31, 2025.

In [ ]:
temporal = ("2022-01-01T00:00:00", "2025-12-31T23:59:59")

### 2.2 Search for Data Granules for AST_08 Data Collection
Here, we perform a query to CMR for AST_08 data that matches our specified spatial and temporal areas of interest using the concept_id for the AST_08 collection. Also, since we are working with a thermal time series we will be using only night acquisitions. To only return night acquisitions, the day night flag is set to night.


In [ ]:
results = earthaccess.search_data(
    concept_id='C3306885674-LPCLOUD',
    bounding_box=bbox,
    temporal=temporal,
    provider='LPCLOUD',
    cloud_cover=(0,1),
    day_night_flag = 'night'
)
print(f'Found {len(results)} Granules')

Here we can see a subset of the URLs for the returned granules.

In [ ]:
urls = [granule.data_links() for granule in results]
urls = [item for sublist in urls for item in sublist]
urls[:6]

## 3. Processing ASTER 08 Data <a id="process"></a>
In this section we set up the `Dask` Processing Environment. This will allow us to parallelize opening and transforming the ASTER 08 data into North-up using reproject. We will then clip the ASTER 08 scenes to our Kilauea Crater area of interest. 

### 3.1 Setup Helper Functions
These functions will allow us to process the ASTER 08 lazily using `Dask`.

Now that we have a list of data URLs, we will configure `GDAL` and `rioxarray` to access the cloud-based assets directly in memory, without the need to download the files.

The Python libraries used to access COG files in Earthdata Cloud leverage GDAL's virtual file systems. The settings below enable GDAL to send authentication information to the `Dask` workers when accessing the ASTER 08 COG files in the Earthdata Cloud and also enable automatic retries in case of network issues. Whether you are running this code in the  cloud or in a local workspace, GDAL configurations must be set in order to successfully access COG files.

In [ ]:
def setup_dask_environment():
    """
    Passes RIO environment variables to dask workers for authentication.
    """

    global env
    cookie_file = os.path.expanduser(f"~/cookies_{os.getpid()}.txt")
    env = rio.Env(
        GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
        GDAL_HTTP_COOKIEFILE=cookie_file,
        GDAL_HTTP_COOKIEJAR=cookie_file,
        GDAL_HTTP_MAX_RETRY="10",
        GDAL_HTTP_RETRY_DELAY="0.5",
    )
    env.__enter__()

Later in our processing, we will combine the data into one `xarray` dataset. We will need the date acquired added to each acquisition's data array to do this. This function will extract the ASTER date acquired timestamp from the band file names.

In [ ]:
def parse_date(tif):
    date_str = tif.split("/")[-1].split(".")[0].split("_")[2][3:]
    date_obj = datetime.strptime(date_str, '%m%d%Y%H%M%S')
    formatted_date = date_obj.strftime('%m%d%Y%H%M%S')
    formatted_date = pd.to_datetime(formatted_date, format = '%m%d%Y%H%M%S')
    
    return formatted_date

This function will allow us to read in the ASTER_08 Surface Kinetic Temperature Data into `rioxarray`. We will load in the three available variables for each acquisition, Surface kinetic temperature composite, QA Data Plane and QA Data Plane 2. The input Cloud Optimized GeoTIFFs (COGS) do not have a nodata value specified in their metadata so we will do that here by setting nodata to zero for the SKT variables and 255 for the QA Data Plane and QA Data Plane 2. We set the nodata value on the QA Data Plane and QA Data Plane 2 to 255 because the QA Data Planes have a data type of 8-bit unsigned integer. Doing this will avoid collisions with the pixel quality value of zero which means good pixel quality in the QA Data Planes.

In [ ]:
def read_rotated(input_file, nodata = 0):
    rotated = rxr.open_rasterio(input_file, mask_and_scale=False)
    nodata_val = 255 if 'QA' in input_file else nodata
    rotated = rotated.rio.write_nodata(nodata_val)

    return rotated

Now we will build a function that will take the rotated ASTER 08 Data and transform the data to a north-up transformation by applying a reprojection and clip the resulting transformed data to our ROI.

In [ ]:
def make_northup(rotated, tif):
    rotated_transform = rotated.rio.transform()
    minx, miny, maxx, maxy = rio.transform.array_bounds(rotated.rio.width, rotated.rio.height, rotated.rio.transform())
    
    res_x = np.sqrt(rotated_transform.a**2 + rotated_transform.b**2)
    res_y = np.sqrt(rotated_transform.d**2 + rotated_transform.e**2)

    out_width = int((maxx - minx)/res_x)
    out_height = int((maxy - miny)/res_y)

    north_up = from_origin(minx, maxy, res_x, res_y)

    north_up_arr = rotated.rio.reproject(rotated.rio.crs, shape = (out_height, out_width), transform = north_up, resampling = Resampling.bilinear)
    
    output_file = tif.split('/')[6][:-4]
    
    if 'QA' in output_file:
        band_name = '_'.join(output_file.split('_')[5:8])
    else:
        band_name = output_file.split('_')[-1]

        
    north_up_arr = north_up_arr.squeeze('band', drop = True)
    date = parse_date(tif)
    
    north_up_arr = north_up_arr.expand_dims(time = [date])
    north_up_arr = north_up_arr.rename(band_name)

    aoi_reproj = aoi.to_crs(north_up_arr.rio.crs)

    try:
        clipped = north_up_arr.rio.clip(aoi_reproj.geometry.values, aoi_reproj.crs, all_touched=True)
    
    except NoDataInBounds                            :
        print(f"No overlap with AOI: {tif}")
        return None

    return clipped



Now we will setup our `Dask` Client. Once the client is set up a link will be provided that we can use to visualize the workers processing the ASTER 08 Data.

In [ ]:
# Initialize Dask Cluster
client = dask.distributed.Client()

# Setup Dask Environment (GDAL Configs)
client.run(setup_dask_environment)

client

### 3.2 Call the Helper Functions and Process Data
Now we are ready to call all the helper functions created in Section 3 to Process our Data. We will use dask.delayed to process each file in parallel across Dask workers. Each worker opens, reprojects, and clips one file independently. Because the source data is rotated, the entire array must be loaded into memory before reprojection, so chunked loading is not used here. The result of this section will be a list of ASTER 08 Data Arrays that are transformed and clipped to our ROI. ***Important: Users should expect this process to take a few minutes to complete.***


In [ ]:
client.restart()

In [ ]:
def process_file(tif):
    rotated = read_rotated(tif)
    clipped = make_northup(rotated, tif)
    
    return clipped

delayed_tasks = [dask.delayed(process_file)(f) for f in urls]
results = dask.compute(*delayed_tasks)

## 4. Post Processing ASTER 08 Data <a id="post"></a>
In this section, we will prepare our list of transformed, clipped ASTER 08 data arrays for visualization. To do this, we will combine the individual data arrays into a single `xarray` Dataset. We will also regrid the data arrays to a common grid, create the dataset, and apply quality filtering.

### 4.1 Filter out arrays that are None values
Since ASTER does not follow a predefined orbit, we may end up with data arrays that contain no valid data. This can happen when a product’s geometry intersects the spatial bounds of our CMR query, but the clipping step in section three removes all valid pixels, leaving an empty array. We need to filter out these empty arrays before constructing our final `xarray` dataset.

In [ ]:
results = tuple(item for item in results if item is not None)
print(len(results), 'Data Arrays Remain')

### 4.2 Grid Data Arrays to Common Grid and Create `xarray` Dataset
Since the input data arrays have many different shapes, we need to establish a common grid before creating the final dataset. Here, we will select the data array with the largest shape to serve as the reference grid and regrid the remaining arrays to match it. Once everything is aligned to this common grid, we can construct the final `xarray` dataset. Lastly, we will apply the scale factor of 0.1 to the Surface Kinetic Temperature data. The result will be a dataset containing 170 ASTER 08 observations.


In [ ]:
def to_dataset(arrays):
    band_arrays = defaultdict(list)
    for da in arrays:
        band_arrays[da.name].append(da)

    # Use the array with the most pixels as the reference grid
    all_das = [da for das in band_arrays.values() for da in das]
    ref = max(all_das, key=lambda da: da.size)

    ds_vars = {}
    for band, das in band_arrays.items():
        aligned = []
        for da in das:
            da = da.rio.reproject_match(ref, resampling=Resampling.nearest)
            aligned.append(da)
        ds_vars[band] = xr.concat(aligned, dim='time').sortby('time')

    return xr.Dataset(ds_vars)

ds = to_dataset(results)
ds['SKT'] = ds['SKT'] * 0.1
ds

### 4.3 Visualize Subset of Results
Here we can create some simple plots using  `matplotlib` to visualize our results. These plots show the measured surface temperature of the Kilauea crater in degrees Kelvin. Notably, two plots in the time series do not cover the entire crater—those with timestamps 2022‑01‑11T08:29:54 and 2022‑01‑27T08:29:34. We will filter these granules out in Section 4.4, since for this use case we only want to retain granules that fully cover the crater.

In [ ]:
n_times = min(6, len(ds.time))
ncols = 6
nrows = 1

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
axes = axes.flatten()

for col in range(n_times):
      data = ds['SKT'].isel(time=col).squeeze().values
      im = axes[col].imshow(data, cmap='YlOrRd')
      axes[col].set_title(str(ds.time[col].values)[:19], fontsize=13)
      axes[col].axis('off')
      cb = plt.colorbar(im, ax=axes[col], shrink=0.8)
      cb.set_label('Kelvin', fontsize=12)

for ax in axes[n_times:]:
      ax.set_visible(False)

plt.tight_layout()
plt.show()

### 4.4 Quality Filtering
This section explains the methodology used to perform quality filtering on the AST_08 data. First, we apply a quality mask: we retain all pixels in the Kinetic Surface Temperature (SKT) variable only when the corresponding pixel in the QA DataPlane is equal to zero. A value of zero in the QA DataPlane indicates a good-quality measurement. For more details on how quality is reported in the ASTER product suite, refer to the [ASTER Quality Plan](https://asterweb.jpl.nasa.gov/content/03_data/04_Documents/ASTER%20QA%20Plan%20v2.0.pdf) documentation.

We then apply an additional quality filter by setting all zero-valued pixels in the SKT variable to NoData. Finally, we remove any arrays whose valid SKT pixel count is less than 75% of the maximum valid count. This step ensures that acquisitions that do not fully cover our region of interest are excluded from the time series.


First, we create a QA mask that identifies pixels in the QA DataPlane with a value of zero, which indicates a good measurement. We then apply this mask to the Surface Kinetic Temperature (SKT) variable. This ensures that only valid pixels are retained, while all others are set to NoData.

In [ ]:
qa_mask = ds['QA_DataPlane'] == 0
ds['SKT'] = ds['SKT'].where(qa_mask)

Second, we set SKT pixels to NaN for acquisitions that do not fully cover our ROI. To do this, we apply a filter that removes any SKT values that do not contain at least 75% good pixels. After applying this filter, we can examine the final dataset.

In [ ]:
ds = ds.where(ds != 0)
valid_counts = [int(ds['SKT'].isel(time=i).notnull().sum()) for i in range(len(ds.time))]
max_valid = max(valid_counts)
valid_indices = [i for i, count in enumerate(valid_counts) if count > 0.75 * max_valid]
ds = ds.isel(time=valid_indices)
ds

Here, we convert the units of the SKT variable from Kelvin to Celsius by applying the formula:
Celsius = Kelvin − 273.15.

In [ ]:
ds['SKT'] = ds['SKT'] - 273.15

Finally, we create a simple visualization to examine the first 12 acquisitions of our time series following quality filtering, showing the Kinetic Surface Temperature of the Kīlauea Crater in degrees Celsius.

In [ ]:
n_times = min(12, len(ds.time))
ncols = 6
nrows = 2

fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))
axes = axes.flatten()

for col in range(n_times):
      data = ds['SKT'].isel(time=col).squeeze().values
      im = axes[col].imshow(data, cmap='YlOrRd')
      axes[col].set_title(str(ds.time[col].values)[:19], fontsize=13)
      axes[col].axis('off')
      cb = plt.colorbar(im, ax=axes[col], shrink=0.8)
      cb.set_label('°C', fontsize=12)

for ax in axes[n_times:]:
      ax.set_visible(False)

plt.tight_layout()
plt.show()

In our granules, we can see evidence of smoke or cloud contamination. Examples include the granules acquired on 2022‑03‑16T08:28:06 and 2022‑04‑01T08:27:34. We can further filter these out, leaving only granules with an unobstructed view of the crater. Granules with a maximum SKT of less than 20 °C will be removed. The remaining acquisitions in the dataset will be used for our final visualizations.

In [ ]:
max_temps = ds['SKT'].max(dim=['x', 'y'])
ds = ds.isel(time=(max_temps >= 20).values)
ds

Following final filtering we are left with 68 observations.

## 5. Visualizations of ASTER 08 Data  <a id="visualizations"></a>
In this section, we explore different ways to extract information from the subset of ASTER 08 Surface Kinetic Temperature data for Kilauea Crater. These visualizations leverage the xarray dataset created in Section 4.
We will generate a time‑series line plot for two latitude–longitude points located around the crater to show how surface temperatures at these locations vary across the filtered time series. In addition, we will overlay the xarray dataset on top of basemap imagery and enable time‑cycling to visualize surface temperature patterns across the entire dataset. The visualizations leverage the display properties of the `holoviews` library.

### 5.1 Create Line Plot
In this section, we create a line plot for two points within the Kilauea Crater. The specified geographic coordinates are reprojected into the coordinate system of the xarray dataset, after which the time series for these locations is extracted and plotted.

First, we need to create a `geopandas` dataframe containing our two points.

In [ ]:
points = {
    'Point_1':(-155.286043, 19.407793),
    'Point_2':(-155.280442, 19.405491)}

sample_points = gpd.GeoDataFrame(
      {'label': list(points.keys())},
      geometry=[Point(lon, lat) for lon, lat in points.values()],
      crs='EPSG:4326')
sample_points

Here, we create a simple plot using `holoviews` to show spatially where our sample points are located.

In [ ]:
tiles = gv.tile_sources.EsriImagery()

pts = gv.Points(sample_points, vdims='label').opts(
      width=600,
      title='ASTER 08 Surface Kinetic Temperature (SKT) Sample Points',
      height=500,
      color='label',       
      cmap='Category10',      
      size=12,
      tools=['hover'],
      legend_position='top_right',
  )

tiles * pts

Here, we sample the Surface Kinetic Temperature variable in our `xarray` dataset at the two selected points to extract the full time series for each location. Because the points are defined in a geographic CRS, we first reproject them into the CRS of the dataset. Once reprojected, we extract the corresponding time‑series values and store them in a `pandas` DataFrame for further examination.

In [ ]:
sample_points_utm = sample_points.to_crs(ds.rio.crs)
time_series = {}
for _, row in sample_points_utm.iterrows():
      ts = ds['SKT'].sel(x=row.geometry.x, y=row.geometry.y, method='nearest')
      time_series[row['label']] = ts.values

df_ts = pd.DataFrame(time_series, index=ds.time.values)
df_ts.columns.name = 'Sample Points'
df_ts

Finally, we create a time‑series line plot showing the Surface Kinetic Temperature (SKT) in Celsius for our two sample points across the full time period.

In [ ]:
df_ts.index = range(len(df_ts))
step = 5

df_ts.hvplot.line(
      x='index',
      y=list(time_series.keys()),
      title='ASTER 08 Surface Kinetic Temperature (SKT) Time Series',
      ylabel='°C',
      xlabel='Acquisition Number',
      width=800,
      height=400,
      colormap='Category10',
  ).opts(
      xrotation=90,
      xticks=[(i, str(t)[:10]) for i, t in enumerate(ds.time.values) if i % step == 0],
  )

### 5.2 Visualize Entire Time Series
Here, we create a map‑based time series. The full `xarray` Surface Kinetic Temperature (SKT) dataset is visualized on a map using `holoviews`. The map is dynamic, allowing the user to change the displayed acquisition using an interactive time slider.


In [ ]:
skt_3857 = ds['SKT'].rio.reproject('EPSG:3857')

height = skt_3857.rio.height
width = skt_3857.rio.width
aspect = width / height

plot_height = 500
plot_width = int(plot_height * aspect)

skt_plot = skt_3857.hvplot(
      x='x', y='y',
      groupby='time',
      cmap='YlOrRd',
      alpha=0.7,
      width=plot_width,
      height=plot_height,
      title='ASTER 08 Surface Kinetic Temperature (SKT) - Kilauea Crater',
      clabel='°C',
      clim=(ds['SKT'].min().item(), ds['SKT'].max().item()),
      colorbar=True,
      data_aspect=1,
  )

tiles * skt_plot

Here we close our `Dask` client, since it is no longer needed.

In [ ]:
client.close()

Success! You have learned how to query, transform, and visualize ASTER 08 data without downloading any source files. You can now replace the collection short name, GeoJSON file, and temporal range in Section 2 with your own inputs and re-run the notebook.

## Contact Info  

Email: LPDAAC@usgs.gov  
Voice: +1-866-573-3222  
Organization: Land Processes Distributed Active Archive Center (LP DAAC)¹  
Website: <https://www.earthdata.nasa.gov/centers/lp-daac>  

¹Work performed under USGS contract G15PD00467 for NASA contract NNG14HH33I.